# 🛡️ Federated Learning Simulation Pipeline (FL-IoT-IDS)
## Distributed Edge Intrusion Detection under Non-IID Skew (Scenarios E2 – E5)
### Master's Thesis: *Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID*
**Author:** Nguyễn Việt Kỳ Quân (2026) | **Institution:** HCMUT - VNU-HCM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/federated_training_pipeline.ipynb)

---

### 🎯 Purpose in Thesis
This notebook provides the core **Federated Learning simulation pipeline** for evaluating decentralized intrusion detection across $K=5$ IoT edge gateways. It directly benchmarks:
- **Scenario E2 (IID Baseline):** Uniform class distribution with standard FedAvg.
- **Scenario E4 (Moderate Non-IID):** Dirichlet $\alpha = 0.5$ label distribution skew.
- **Scenario E5 (Severe Non-IID & FedProx):** Extreme Dirichlet $\alpha = 0.1$ label skew with proximal regularization parameter $\mu = 0.05$ to penalize **Client Drift**.

Powered by modular components from `src.data.partitioner`, `src.training.federated_trainer`, `src.federated.strategy`, and `src.evaluation`.

---
### 🛠️ Step 1: Environment Setup & Hardware Acceleration
We check compute resources, synchronize repository code in Colab, and enforce deterministic seeds.

In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Compute Device: {device} | PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'   Active GPU: {torch.cuda.get_device_name(0)}')

# Colab setup
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    repo_dir = '/content/FedLearning'
    if not os.path.exists(repo_dir):
        subprocess.run(['git', 'clone', 'https://github.com/NVKQ2022/FedLearning.git', repo_dir], check=True)
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
elif os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.utils.seed import set_seed
set_seed(42)
print('✅ Deterministic random seed locked at 42.')

---
### ⚙️ Step 2: Global Experiment Configuration
Select the scenario to execute (`scenario_name`, `partition_type`, `dirichlet_alpha`, `federated_strategy`, and proximal coefficient `mu`).

In [ ]:
from src.utils.config import ExperimentConfig

# ==============================================================================
# 🛠️ CONFIGURE EXPERIMENTAL SCENARIO HERE
# - Scenario E2: federated_strategy='fedavg',  partition_type='iid',       mu=0.0
# - Scenario E4: federated_strategy='fedavg',  partition_type='dirichlet', dirichlet_alpha=0.5, mu=0.0
# - Scenario E5: federated_strategy='fedprox', partition_type='dirichlet', dirichlet_alpha=0.1, mu=0.05
# ==============================================================================
CONFIG = ExperimentConfig(
    experiment_name="federated_e2_fedavg_iid",
    seed=42,
    
    # Dataset & Preprocessing
    sample_size=100000,                 # 100k samples for rapid execution
    test_size=0.2,                      # 20% holdout test set (unseen by all clients)
    val_size=0.1,                       # 10% centralized server validation set
    scaler_type="robust",               # 'robust' (median/IQR)
    batch_size=64,                      # Local batch size per client
    
    # Architecture & Regularization
    hidden_dims=(128, 64),              # Lightweight edge MLP backbone (~15k params)
    dropout_rate=0.2,
    
    # Loss Function
    loss_type="focal_loss",             # MultiClassFocalLoss
    class_weight_strategy="balanced",   # Inverse frequency class weighting
    focal_gamma=2.0,
    
    # Client Optimization
    optimizer_type="adamw",
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_grad_norm=5.0,
    
    # Federated Parameters
    num_clients=5,                      # Number of edge clients K
    num_rounds=10,                      # Server communication rounds T
    local_epochs=2,                     # Client local epochs per round E
    federated_strategy="fedavg",        # 'fedavg' (default) or 'fedprox'
    mu=0.0,                             # FedProx proximal coefficient (0.0 for FedAvg, 0.05 for FedProx)
    partition_type="iid",               # 'iid' or 'dirichlet'
    dirichlet_alpha=0.5                 # Dirichlet concentration parameter (0.5 moderate, 0.1 severe)
)

CONFIG.display()


---
### 💾 Step 3: Leak-Free Preprocessing & Edge Client Partitioning
We preprocess the dataset and distribute the training set across $K=5$ client partitions according to `CONFIG.partition_type`.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
from src.data.partitioner import (
    StratifiedIIDPartitioner,
    DirichletNonIIDPartitioner,
    create_client_dataloaders,
    summarize_client_partitions
)
from src.eda import plot_partition_heterogeneity

# Locate dataset
candidate_paths = [
    CONFIG.gdrive_csv,
    CONFIG.alt_gdrive_csv,
    CONFIG.target_csv_path,
    'datasets/CICIOT2023/merged_CICIOT2023_data.csv',
    'datasets/merged_CICIOT2023_data.csv',
]
csv_path = next((p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 10*1024*1024), None)
if not csv_path:
    csv_path = CONFIG.target_csv_path

print(f"📁 Loading and preprocessing from: {csv_path}")
data = load_and_preprocess_ciciot2023(
    csv_path=csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(data['y_train'], data['num_classes'], strategy=CONFIG.class_weight_strategy)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

# Partition data across clients
if CONFIG.partition_type == "iid":
    partitioner = StratifiedIIDPartitioner(num_clients=CONFIG.num_clients, seed=CONFIG.seed)
else:
    partitioner = DirichletNonIIDPartitioner(
        num_clients=CONFIG.num_clients,
        alpha=CONFIG.dirichlet_alpha,
        min_samples_per_client=100,
        seed=CONFIG.seed
    )

client_partitions = partitioner.partition(data['X_train'], data['y_train'])
client_loaders = create_client_dataloaders(data['X_train'], data['y_train'], client_partitions, batch_size=CONFIG.batch_size)

# Display cross-client distribution summary
summary_df = summarize_client_partitions(client_partitions, data['y_train'], class_names=class_names)
display(summary_df[['client_id', 'total_samples', 'dominant_class', 'dominant_pct', 'entropy']])

# Render stacked heterogeneity plot
plot_partition_heterogeneity(
    summary_df=summary_df,
    class_names=class_names,
    title=f"Client Label Distribution ({CONFIG.partition_type.upper()} alpha={CONFIG.dirichlet_alpha})",
    show=True
)

---
### 🏛️ Step 4: Server Strategy & Client Model Setup
We initialize the global `TabularIoTMLPModel`, loss function with class weights, and server aggregation strategy (`FedAvgStrategy` or `FedProxStrategy`).

In [ ]:
from src.models.tabular_mlp import TabularIoTMLPModel
from src.losses.focal_loss import build_loss_function
from src.federated.strategy import FedAvgStrategy, FedProxStrategy

# 1. Initialize Global Model
global_model = TabularIoTMLPModel(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)

# 2. Build Loss Function
criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights if CONFIG.class_weight_strategy != 'none' else None,
    gamma=CONFIG.focal_gamma,
    device=device
)

# 3. Select Server Aggregation Strategy
if CONFIG.federated_strategy.lower() == "fedprox":
    strategy = FedProxStrategy(mu=CONFIG.mu)
else:
    strategy = FedAvgStrategy()

print(f"✅ Initialized Global Model: {global_model.get_num_parameters():,} parameters ({global_model.get_model_size_kb():.2f} KB)")
print(f"✅ Active Server Strategy:   {strategy.name} (mu={getattr(strategy, 'mu', 0.0)})")

---
### 🚀 Step 5: Multi-Round Federated Simulation
We execute the federated training loop across $T$ rounds:
1. Server broadcasts global weights $w_t$ to clients.
2. Clients execute local training with optional proximal penalty $\frac{\mu}{2} \|w - w_t\|^2$.
3. Server aggregates client updates using sample-weighted FedAvg / FedProx.
4. Global model is evaluated on the holdout validation set.

In [ ]:
import copy, time
from src.optimizers.optimizer import build_optimizer
from src.training.federated_trainer import FederatedClientTrainer
from src.evaluation.evaluator import evaluate_comprehensive

print(f"Starting Federated Simulation: {CONFIG.num_clients} clients, {CONFIG.num_rounds} rounds, {CONFIG.local_epochs} local epochs...")
print(f"Strategy: {strategy.name} | Proximal mu: {CONFIG.mu if CONFIG.federated_strategy == 'fedprox' else 0.0}")
print("=" * 80)

round_history = {
    "round": [],
    "train_loss": [],
    "val_loss": [],
    "val_macro_f1": [],
    "drift_l2": []
}
best_val_macro_f1 = -1.0
best_round = 0
best_weights = None
current_mu = CONFIG.mu if CONFIG.federated_strategy.lower() == "fedprox" else 0.0

sim_start = time.perf_counter()

for round_idx in range(1, CONFIG.num_rounds + 1):
    client_results = []
    round_drifts = []
    
    # 1. Local Training Phase
    for client_id in range(CONFIG.num_clients):
        local_model = copy.deepcopy(global_model)
        opt = build_optimizer(local_model, CONFIG.optimizer_type, lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
        trainer_c = FederatedClientTrainer(local_model, opt, criterion, device=device, max_grad_norm=CONFIG.max_grad_norm)
        
        loss_c, acc_c = trainer_c.train_epochs(
            dataloader=client_loaders[client_id],
            num_epochs=CONFIG.local_epochs,
            global_model=global_model,
            mu=current_mu
        )
        
        # Calculate parameter drift: ||w_local - w_global||_2
        drift = sum(
            torch.norm(p_loc - p_glob).item() ** 2
            for p_loc, p_glob in zip(local_model.parameters(), global_model.parameters())
        ) ** 0.5
        round_drifts.append(drift)
        
        n_c = len(client_partitions[client_id])
        client_results.append((local_model.get_weights(), n_c, {"loss": loss_c, "acc": acc_c}))
    
    # 2. Server Aggregation Phase
    agg_weights, agg_metrics = strategy.aggregate_fit(round_idx, client_results)
    global_model.set_weights(agg_weights)
    
    # 3. Server Validation Phase
    val_metrics = evaluate_comprehensive(global_model, data['val_loader'], class_names, criterion=criterion, device=device)
    
    v_f1 = val_metrics["macro_f1"]
    v_loss = val_metrics["loss"]
    t_loss = agg_metrics.get("train_loss", 0.0)
    avg_drift = float(np.mean(round_drifts))
    
    round_history["round"].append(round_idx)
    round_history["train_loss"].append(t_loss)
    round_history["val_loss"].append(v_loss)
    round_history["val_macro_f1"].append(v_f1)
    round_history["drift_l2"].append(avg_drift)
    
    is_best = v_f1 > best_val_macro_f1
    if is_best:
        best_val_macro_f1 = v_f1
        best_round = round_idx
        best_weights = copy.deepcopy(global_model.get_weights())
        
    star = "⭐ (Best)" if is_best else ""
    print(f"[Round {round_idx:02d}/{CONFIG.num_rounds:02d}] Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Macro-F1: {v_f1*100:5.2f}% | Drift L2: {avg_drift:.4f} {star}")

if best_weights is not None:
    global_model.set_weights(best_weights)

total_time = time.perf_counter() - sim_start
print("=" * 80)
print(f"🎉 Simulation Completed in {total_time:.1f}s ({total_time/60:.2f} min) | Best Round: {best_round:02d} (Val Macro-F1: {best_val_macro_f1*100:.2f}%)")

---
### 📈 Step 6: Multi-Round Convergence & Parameter Drift
We inspect the Validation Macro-F1 trajectory and client parameter drift $\|w_{\text{local}} - w_t\|_2$ across rounds.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=120)
rounds = round_history["round"]

# Left: Macro-F1 Trajectory
ax1.plot(rounds, [f1 * 100 for f1 in round_history["val_macro_f1"]], 'o-', color='#2980b9', linewidth=2)
ax1.set_title(f"Validation Macro-F1 ({CONFIG.experiment_name})", fontsize=11, fontweight='bold')
ax1.set_xlabel("Communication Round", fontsize=10)
ax1.set_ylabel("Macro-F1 (%)", fontsize=10)
ax1.grid(True, linestyle="--", alpha=0.5)

# Right: Parameter Drift L2 Norm
ax2.plot(rounds, round_history["drift_l2"], 's-', color='#e74c3c', linewidth=2)
ax2.set_title("Average Client Parameter Drift ||w - w_t||_2", fontsize=11, fontweight='bold')
ax2.set_xlabel("Communication Round", fontsize=10)
ax2.set_ylabel("L2 Drift Norm", fontsize=10)
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

---
### 🏆 Step 7: Holdout Test Evaluation
We benchmark the restored best global model on the stationary holdout test set (20% of data, zero leakage).


In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive
from src.visualization import plot_confusion_matrix

test_results = evaluate_comprehensive(
    model=global_model,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print('=' * 65)
print(f"🏆 {CONFIG.experiment_name.upper()} FINAL HOLDOUT TEST EVALUATION")
print('=' * 65)
print(f"Test Accuracy:    {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-F1 Score:   {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted-F1:      {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:  {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:     {test_results['macro_recall'] * 100:.2f}%")
print('-' * 65)
print('Minority Class Recall (Stealth Infiltration Intrusions):')
for cls, rec in test_results['minority_recall'].items():
    print(f"   • {cls:<15}: {rec * 100:.2f}%")
print('=' * 65)

# Render Confusion Matrix
plot_confusion_matrix(
    cm=test_results['confusion_matrix'],
    class_names=class_names,
    title=f"{CONFIG.experiment_name}: Normalized Confusion Matrix",
    show=True
)

---
### 💾 Step 8: Export Scenario Results Hierarchy
We persist the complete scenario results under `scenarios/<scenario_name>/` using our standardized scenario hierarchy (client partitions, client EDA profiles, and server round metrics).

In [ ]:
from src.federated.scenario import setup_scenario_hierarchy
import json

scenario_dir = f"scenarios/{CONFIG.experiment_name}"
setup_scenario_hierarchy(
    scenario_name=CONFIG.experiment_name,
    client_partitions=client_partitions,
    X=data['X_train'],
    y=data['y_train'],
    class_names=class_names,
    scenarios_root="scenarios"
)

# Export final server evaluation metrics
metrics_export = {k: v for k, v in test_results.items() if k != 'confusion_matrix'}
metrics_export['round_history'] = round_history
metrics_path = f"{scenario_dir}/server/metrics.json"

with open(metrics_path, 'w') as f:
    json.dump(metrics_export, f, indent=2)

print(f"✅ Successfully exported scenario artifacts to: {scenario_dir}/")
print(f"   - Client Partitions: {scenario_dir}/client_*/partition.npz")
print(f"   - Client EDA Profiles: {scenario_dir}/client_*/eda.json")
print(f"   - Server Evaluation: {metrics_path}")

print(f"\n🎉 {CONFIG.experiment_name} simulation completed and certified!")